<a href="https://colab.research.google.com/github/prometricas/William_Rondon/blob/main/COCTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cálculos inferenciales COCTS – sección 4.4.1

Notebook para obtener las tablas estadísticas de la sección **4.4.1. Análisis inferencial en las concepciones acerca de la NDC (COCTS)**.  
No genera gráficos. Solo produce tablas para normalidad, comparación entre grupos, cambios intragrupo y contraste de cambios.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from IPython.display import display

# Yo cargo el archivo validado; si no está en Colab, pido subirlo.
archivo = "data_Quimica_validado.xlsx"
try:
    cocts = pd.read_excel(archivo, sheet_name="COCTS")
except FileNotFoundError:
    from google.colab import files
    subidos = files.upload()
    archivo = list(subidos.keys())[0]
    cocts = pd.read_excel(archivo, sheet_name="COCTS")

cocts.head()

Saving data_Quimica_validado.xlsx to data_Quimica_validado.xlsx


,momento,grupo,curso,id_estudiante,nombre_estudiante,sexo,edad,q_10111_definicion_ciencia,q_10411_interdependencia_cyt,q_20141_influencia_gobierno,...,q_90211_modelos_cientificos,q_90311_esquemas_clasificacion,q_90411_provisionalidad,q_90651_papel_error,q_91011_estatus_epistemologico,d1_definiciones_cyt,d2_sociologia_externa,d3_sociologia_interna,d4_epistemologia,cocts_total_ponderado
0,Pretest,Control,1004,B_Est 2,Ana Mileth Collo Valencia,F,16,0.100000,0.208333,-0.229167,...,0.222222,0.097222,-0.208333,-0.277778,-0.305556,0.154167,0.123264,-0.137153,-0.094444,-0.014630
1,Postest,Control,1004,B_Est 2,Ana Mileth Collo Valencia,F,16,-0.250000,0.625000,0.375000,...,-0.361111,-0.361111,0.041667,0.250000,-0.611111,0.187500,0.034722,0.003472,-0.208333,-0.034259
2,Pretest,Control,1004,B_Est 3,Pineda Caro Liz,F,15,0.141667,0.166667,0.625000,...,-0.555556,-0.125000,0.250000,-0.277778,-0.041667,0.154167,0.180556,0.008681,-0.150000,0.021019
3,Postest,Control,1004,B_Est 3,Pineda Caro Liz,F,15,0.708333,0.208333,0.625000,...,0.472222,0.069444,0.208333,0.333333,0.055556,0.458333,0.437500,0.302083,0.227778,0.334259
4,Pretest,Control,1004,B_Est 4,Anny Jireth Cabrera Calderon,F,17,0.291667,-0.041667,0.104167,...,0.152778,0.486111,0.125000,-0.069444,-0.027778,0.125000,0.199653,0.071181,0.133333,0.133333


In [2]:
# Yo defino solo las variables necesarias para la sección 4.4.1.
variables = {
    "Definiciones de CyT": "d1_definiciones_cyt",
    "Sociología externa": "d2_sociologia_externa",
    "Sociología interna": "d3_sociologia_interna",
    "Epistemología": "d4_epistemologia",
    "Índice ponderado global": "cocts_total_ponderado"
}

grupos = ["Control", "Experimental"]
momentos = ["Pretest", "Postest"]

# Yo verifico que la hoja tenga los grupos y momentos requeridos.
print(pd.crosstab(cocts["grupo"], cocts["momento"]))

momento       Postest  Pretest
grupo                         
Control            24       24
Experimental       27       27


In [3]:
# Yo preparo funciones breves para redondeo y tamaños del efecto.
def r3(x):
    return np.nan if pd.isna(x) else round(float(x), 3)

def p3(p):
    if pd.isna(p):
        return ""
    return "< 0.001" if p < 0.001 else f"{p:.3f}"

def d_independiente(x, y):
    n1, n2 = len(x), len(y)
    gl = n1 + n2 - 2
    sp = np.sqrt(((n1 - 1) * np.var(x, ddof=1) + (n2 - 1) * np.var(y, ddof=1)) / gl)
    return (np.mean(x) - np.mean(y)) / sp

def dz_pareado(pre, post):
    dif = post - pre
    return np.mean(dif) / np.std(dif, ddof=1)

def normalidad(momento):
    filas = []
    datos = cocts[cocts["momento"] == momento]
    for nombre, col in variables.items():
        fila = {"Variable": nombre}
        for grupo in grupos:
            x = datos.loc[datos["grupo"] == grupo, col].dropna()
            w, p = stats.shapiro(x)
            fila[f"W {grupo}"] = r3(w)
            fila[f"p {grupo}"] = p3(p)
        filas.append(fila)
    return pd.DataFrame(filas)

def comparacion_independiente_t(momento):
    filas = []
    datos = cocts[cocts["momento"] == momento]
    for nombre, col in variables.items():
        control = datos.loc[datos["grupo"] == "Control", col].dropna().to_numpy()
        experimental = datos.loc[datos["grupo"] == "Experimental", col].dropna().to_numpy()
        f_lev, p_lev = stats.levene(control, experimental, center="mean")
        t, p = stats.ttest_ind(control, experimental, equal_var=(p_lev >= 0.05))
        if p_lev >= 0.05:
            gl = len(control) + len(experimental) - 2
        else:
            s1, s2 = np.var(control, ddof=1), np.var(experimental, ddof=1)
            n1, n2 = len(control), len(experimental)
            gl = (s1/n1 + s2/n2)**2 / ((s1/n1)**2/(n1-1) + (s2/n2)**2/(n2-1))
        filas.append({
            "Variable": nombre,
            "F de Levene": r3(f_lev),
            "p Levene": p3(p_lev),
            "t": r3(t),
            "gl": r3(gl),
            "p": p3(p),
            "Diferencia de medias (Control - Experimental)": r3(np.mean(control) - np.mean(experimental)),
            "d": r3(d_independiente(control, experimental))
        })
    return pd.DataFrame(filas)

def cambios_intragrupo():
    filas = []
    for grupo in grupos:
        datos = cocts[cocts["grupo"] == grupo]
        for nombre, col in variables.items():
            pre = datos.loc[datos["momento"] == "Pretest", ["id_estudiante", col]].rename(columns={col: "pre"})
            post = datos.loc[datos["momento"] == "Postest", ["id_estudiante", col]].rename(columns={col: "post"})
            pares = pre.merge(post, on="id_estudiante").dropna()
            t, p = stats.ttest_rel(pares["post"], pares["pre"])
            filas.append({
                "Grupo": grupo,
                "Variable": nombre,
                "Pretest": r3(pares["pre"].mean()),
                "Postest": r3(pares["post"].mean()),
                "Diferencia (post-pre)": r3((pares["post"] - pares["pre"]).mean()),
                "t": r3(t),
                "gl": len(pares) - 1,
                "p": p3(p),
                "dz": r3(dz_pareado(pares["pre"], pares["post"]))
            })
    return pd.DataFrame(filas)

def contraste_cambios():
    filas = []
    for nombre, col in variables.items():
        cambios = {}
        for grupo in grupos:
            datos = cocts[cocts["grupo"] == grupo]
            pre = datos.loc[datos["momento"] == "Pretest", ["id_estudiante", col]].rename(columns={col: "pre"})
            post = datos.loc[datos["momento"] == "Postest", ["id_estudiante", col]].rename(columns={col: "post"})
            pares = pre.merge(post, on="id_estudiante").dropna()
            cambios[grupo] = (pares["post"] - pares["pre"]).to_numpy()
        t, p = stats.ttest_ind(cambios["Experimental"], cambios["Control"], equal_var=True)
        filas.append({
            "Variable": nombre,
            "Cambio experimental": r3(np.mean(cambios["Experimental"])),
            "Cambio control": r3(np.mean(cambios["Control"])),
            "Diferencia de cambios (Experimental - Control)": r3(np.mean(cambios["Experimental"]) - np.mean(cambios["Control"])),
            "t": r3(t),
            "gl": len(cambios["Experimental"]) + len(cambios["Control"]) - 2,
            "p": p3(p),
            "d": r3(d_independiente(cambios["Experimental"], cambios["Control"]))
        })
    return pd.DataFrame(filas)

def alertas_supuestos():
    filas = []
    for momento in momentos:
        nrm = normalidad(momento)
        for _, fila in nrm.iterrows():
            for grupo in grupos:
                ptxt = fila[f"p {grupo}"]
                if ptxt == "< 0.001" or (ptxt and float(ptxt) < 0.05):
                    filas.append({"Momento": momento, "Variable": fila["Variable"], "Grupo": grupo, "Alerta": "Shapiro-Wilk p < 0.05"})
    for momento in momentos:
        datos = cocts[cocts["momento"] == momento]
        for nombre, col in variables.items():
            control = datos.loc[datos["grupo"] == "Control", col].dropna()
            experimental = datos.loc[datos["grupo"] == "Experimental", col].dropna()
            _, p_lev = stats.levene(control, experimental, center="mean")
            if p_lev < 0.05:
                filas.append({"Momento": momento, "Variable": nombre, "Grupo": "Control/Experimental", "Alerta": "Levene p < 0.05"})
    return pd.DataFrame(filas)


In [4]:
# Yo calculo las tablas de normalidad para el pretest y el postest.
tabla_15_normalidad_pretest = normalidad("Pretest")
tabla_17_normalidad_postest = normalidad("Postest")

display(tabla_15_normalidad_pretest)
display(tabla_17_normalidad_postest)

,Variable,W Control,p Control,W Experimental,p Experimental
0,Definiciones de CyT,0.954,0.325,0.984,0.934
1,Sociología externa,0.975,0.792,0.975,0.734
2,Sociología interna,0.967,0.600,0.974,0.709
3,Epistemología,0.931,0.105,0.956,0.295
4,Índice ponderado global,0.955,0.354,0.990,0.993


,Variable,W Control,p Control,W Experimental,p Experimental
0,Definiciones de CyT,0.983,0.937,0.975,0.745
1,Sociología externa,0.977,0.837,0.984,0.933
2,Sociología interna,0.957,0.387,0.964,0.451
3,Epistemología,0.940,0.165,0.878,0.004
4,Índice ponderado global,0.971,0.684,0.985,0.947


In [5]:
# Yo calculo las comparaciones entre grupos con prueba t y d de Cohen.
tabla_16_intergrupal_pretest = comparacion_independiente_t("Pretest")
tabla_18_intergrupal_postest = comparacion_independiente_t("Postest")

display(tabla_16_intergrupal_pretest)
display(tabla_18_intergrupal_postest)

,Variable,F de Levene,p Levene,t,gl,p,Diferencia de medias (Control - Experimental),d
0,Definiciones de CyT,0.500,0.483,-0.672,49.0,0.505,-0.028,-0.188
1,Sociología externa,0.074,0.787,1.463,49.0,0.150,0.043,0.411
2,Sociología interna,0.364,0.549,0.906,49.0,0.369,0.033,0.254
3,Epistemología,0.665,0.419,0.462,49.0,0.646,0.019,0.130
4,Índice ponderado global,0.339,0.563,0.983,49.0,0.330,0.023,0.276


,Variable,F de Levene,p Levene,t,gl,p,Diferencia de medias (Control - Experimental),d
0,Definiciones de CyT,0.762,0.387,-1.542,49.0,0.130,-0.069,-0.432
1,Sociología externa,1.373,0.247,1.693,49.0,0.097,0.045,0.475
2,Sociología interna,0.238,0.628,0.159,49.0,0.874,0.005,0.045
3,Epistemología,1.582,0.214,-0.480,49.0,0.633,-0.017,-0.135
4,Índice ponderado global,2.058,0.158,-0.057,49.0,0.955,-0.001,-0.016


In [6]:
# Yo calculo los cambios del pretest al postest dentro de cada grupo.
tabla_19_cambios_intragrupo = cambios_intragrupo()
display(tabla_19_cambios_intragrupo)

,Grupo,Variable,Pretest,Postest,Diferencia (post-pre),t,gl,p,dz
0,Control,Definiciones de CyT,0.176,0.220,0.044,1.051,23,0.304,0.214
1,Control,Sociología externa,0.219,0.252,0.033,1.079,23,0.292,0.220
2,Control,Sociología interna,0.131,0.174,0.042,1.580,23,0.128,0.323
3,Control,Epistemología,0.153,0.172,0.019,0.596,23,0.557,0.122
4,Control,Índice ponderado global,0.168,0.200,0.032,1.597,23,0.124,0.326
5,Experimental,Definiciones de CyT,0.204,0.289,0.085,2.024,26,0.053,0.389
6,Experimental,Sociología externa,0.175,0.206,0.031,1.451,26,0.159,0.279
7,Experimental,Sociología interna,0.099,0.169,0.070,2.211,26,0.036,0.426
8,Experimental,Epistemología,0.134,0.189,0.055,1.947,26,0.062,0.375
9,Experimental,Índice ponderado global,0.145,0.202,0.057,4.285,26,< 0.001,0.825


In [7]:
# Yo comparo los cambios pretest-postest entre el grupo experimental y el grupo control.
tabla_20_contraste_cambios = contraste_cambios()
display(tabla_20_contraste_cambios)

,Variable,Cambio experimental,Cambio control,Diferencia de cambios (Experimental - Control),t,gl,p,d
0,Definiciones de CyT,0.085,0.044,0.041,0.687,49,0.495,0.193
1,Sociología externa,0.031,0.033,-0.002,-0.056,49,0.955,-0.016
2,Sociología interna,0.070,0.042,0.028,0.661,49,0.512,0.186
3,Epistemología,0.055,0.019,0.036,0.850,49,0.400,0.238
4,Índice ponderado global,0.057,0.032,0.024,1.030,49,0.308,0.289


In [8]:
# Yo reviso si algún supuesto estadístico requiere atención antes de redactar.
tabla_alertas = alertas_supuestos()
if tabla_alertas.empty:
    print("No se detectaron alertas en Shapiro-Wilk ni en Levene con alfa = 0.05.")
else:
    display(tabla_alertas)

,Momento,Variable,Grupo,Alerta
0,Postest,Epistemología,Experimental,Shapiro-Wilk p < 0.05


In [9]:
# Yo exporto las tablas a Excel para copiarlas con facilidad en Word.
salida = "tablas_COCTS_4_4_1.xlsx"
with pd.ExcelWriter(salida) as writer:
    tabla_15_normalidad_pretest.to_excel(writer, sheet_name="T15_normalidad_pre", index=False)
    tabla_17_normalidad_postest.to_excel(writer, sheet_name="T17_normalidad_post", index=False)
    tabla_16_intergrupal_pretest.to_excel(writer, sheet_name="T16_inter_pre", index=False)
    tabla_18_intergrupal_postest.to_excel(writer, sheet_name="T18_inter_post", index=False)
    tabla_19_cambios_intragrupo.to_excel(writer, sheet_name="T19_cambios_intra", index=False)
    tabla_20_contraste_cambios.to_excel(writer, sheet_name="T20_contraste_cambios", index=False)
    tabla_alertas.to_excel(writer, sheet_name="Alertas_supuestos", index=False)

print(f"Archivo generado: {salida}")

Archivo generado: tablas_COCTS_4_4_1.xlsx
